In [1]:

import os, json, random, math
from collections import defaultdict
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score

try:
    from torch_geometric.data import Data
    from torch_geometric.loader import NeighborLoader
    from torch_geometric.nn import RGCNConv, GATConv
except Exception as e:
    raise ImportError("Install torch_geometric and dependencies. Error: {}".format(e))

print("Torch:", torch.__version__)


c:\Users\Manasa\OneDrive\Desktop\Drug_Repurposing_Gnn\Drug_Repurposing_Gnn\.venv\lib\site-packages\torch_geometric\typing.py:31: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
c:\Users\Manasa\OneDrive\Desktop\Drug_Repurposing_Gnn\Drug_Repurposing_Gnn\.venv\lib\site-packages\torch_geometric\typing.py:42: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "


ImportError: Install torch_geometric and dependencies. Error: [WinError 127] The specified procedure could not be found

In [2]:

GRAPH_DIR = "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph"    # folder with edge_index.pt, edge_type.pt, entities.txt, train_inductive.csv, ...
OUT_DIR = "/mnt/data/drkg_runs"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
EPOCHS = 8
BATCH_SIZE = 1024
NEIGHBOR_SAMPLES = [20, 10]   # neighbors per layer for NeighborLoader (2-layer GNN example)
HDIM = 128
OUTDIM = 128
LR = 1e-3
WEIGHT_DECAY = 1e-6
DROPOUT = 0.3
NEG_RATIO = 1
PATH_REG_WEIGHT = 0.5
ENSEMBLE_SIZE = 2     # how many re-trains per model for ensemble
MC_RUNS = 30
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# relation name sets to detect mechanistic edges; edit if your relation names differ
COMPOUND_GENE_RELATIONS = set(["targets", "binds_to", "interacts_with", "targets_gene", "targets_gene_by"])
GENE_DISEASE_RELATIONS = set(["associated_with", "associates", "gene_associated_disease", "gene_association"])
DRUG_DISEASE_RELATIONS = set(["treats", "indicated_for", "has_indication", "therapeutic_for"])



Device: cuda


In [3]:
# -------------------------
# Utilities
# -------------------------
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def safe_read_lines(path):
    if not os.path.exists(path):
        return []
    with open(path, 'r', encoding='utf-8') as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    return lines

def infer_entity_type_str(s: str):
    s_low = s.lower()
    if s_low.startswith("compound") or s_low.startswith("drug") or s_low.startswith("db") or "drug" in s_low or "compound" in s_low:
        return "compound"
    if s_low.startswith("gene") or s_low.startswith("hgnc:") or "entrez" in s_low or (s.isupper() and len(s)<=8 and s.isalpha()):
        return "gene"
    if s_low.startswith("disease") or "disease" in s_low or s_low.startswith("doid") or s_low.startswith("mesh") or "phenotype" in s_low:
        return "disease"
    # fallback: if has pipe-separated metadata, try to detect a type token
    parts = s.replace("|", "\t").split("\t")
    for p in parts:
        pl = p.lower()
        if pl in ("compound","drug","gene","disease"): return pl
    return "other"

def try_parse_entities(entities_lines: List[str]):
    """
    Support formats:
    - one id per line (entity ID)
    - id \t name \t type
    - id,type (csv)
    - JSON lines (rare)
    """
    ent_list = []
    for ln in entities_lines:
        if "\t" in ln:
            parts = ln.split("\t")
            ent_list.append(parts[0].strip())
        elif "," in ln and not ln.startswith("http"):
            # maybe CSV like id,name,type
            parts = ln.split(",")
            ent_list.append(parts[0].strip())
        else:
            ent_list.append(ln.strip())
    return ent_list



In [10]:
# -------------------------
# Load graph files
# -------------------------
edge_index_path = os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/edge_index.pt")
edge_type_path = os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/edge_type.pt")
entities_path = os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/entities.txt")
relations_path = os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/relations.txt")

assert os.path.exists(edge_index_path), f"{edge_index_path} missing"
assert os.path.exists(edge_type_path), f"{edge_type_path} missing"

edge_index = torch.load(edge_index_path)
edge_type = torch.load(edge_type_path)

entities_lines = safe_read_lines(entities_path)
entities = try_parse_entities(entities_lines)
num_nodes = len(entities) if len(entities)>0 else int(edge_index.max().item())+1
print("Num nodes:", num_nodes, "entities parsed:", len(entities))

# build mapping str->gid (if entities.txt contains canonical IDs in same format as your inductive CSV)
ent2gid = {entities[i]: i for i in range(len(entities))} if len(entities)>0 else {}
# fallback: user may have triples using integer ids already; we'll handle that during mapping

# build PyG Data object
data = Data()
data.num_nodes = num_nodes
data.edge_index = edge_index
data.edge_type = edge_type  # optional; used in RGCN
# node features: learnable embeddings will be used inside models, so no x required here



Num nodes: 94046 entities parsed: 94046


In [11]:
# -------------------------
# Load inductive CSVs (train/val/test)
# -------------------------
def read_triples_csv(path):
    if not os.path.exists(path):
        return []
    df = pd.read_csv(path, header=None)
    if df.shape[1] >= 3:
        triples = [(str(r[0]).strip(), str(r[1]).strip(), str(r[2]).strip()) for r in df.values]
    else:
        triples = []
    return triples

train_csv = os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/train_inductive.csv")
val_csv = os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/val_inductive.csv")
test_csv = os.path.join(GRAPH_DIR, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/test_inductive.csv")
train_triples = read_triples_csv(train_csv)
val_triples = read_triples_csv(val_csv)
test_triples = read_triples_csv(test_csv)
print("Loaded inductive CSV triples:", len(train_triples), len(val_triples), len(test_triples))



Loaded inductive CSV triples: 5261828 584649 27787


In [6]:
# helper to map triple string IDs to gids (tolerant)
def map_triple_to_gid(triple, ent_map):
    h, r, t = triple
    # direct match
    if h in ent_map and t in ent_map:
        return (ent_map[h], r, ent_map[t])
    # try numeric
    try:
        hi = int(h); ti = int(t)
        if 0 <= hi < data.num_nodes and 0 <= ti < data.num_nodes:
            return (hi, r, ti)
    except:
        pass
    # fuzzy match: suffix/prefix
    for key in ent_map:
        if key.endswith(h) or h.endswith(key):
            h = key; break
    for key in ent_map:
        if key.endswith(t) or t.endswith(key):
            t = key; break
    if h in ent_map and t in ent_map:
        return (ent_map[h], r, ent_map[t])
    return None

train_gid_triples = [map_triple_to_gid(t, ent2gid) for t in train_triples]
train_gid_triples = [t for t in train_gid_triples if t is not None]
val_gid_triples = [map_triple_to_gid(t, ent2gid) for t in val_triples]; val_gid_triples = [t for t in val_gid_triples if t is not None]
test_gid_triples = [map_triple_to_gid(t, ent2gid) for t in test_triples]; test_gid_triples = [t for t in test_gid_triples if t is not None]
print("Mapped triples -> gids:", len(train_gid_triples), len(val_gid_triples), len(test_gid_triples))



Mapped triples -> gids: 5253176 583712 27786


In [12]:
# -------------------------
# Infer entity types (compound/gene/disease) by heuristics or from entities.txt if it had type info
# -------------------------
ent_type_map = {}
if len(entities_lines) > 0:
    # attempt to detect 'id \t name \t type' or 'id,type'
    for i, ln in enumerate(entities_lines):
        parts_tab = ln.split("\t")
        parts_comma = ln.split(",")
        if len(parts_tab) >= 3:
            # assume id\tname\ttype
            ent_id = parts_tab[0].strip()
            typ = parts_tab[-1].strip().lower()
            ent_type_map[i] = typ if typ in ("compound","gene","disease") else infer_entity_type_str(parts_tab[0])
        elif len(parts_comma) >= 2 and parts_comma[-1].strip().lower() in ("compound","gene","disease"):
            ent_type_map[i] = parts_comma[-1].strip().lower()
        else:
            ent_type_map[i] = infer_entity_type_str(entities[i])
else:
    # no entity names; fallback by other heuristics (not great)
    for gid in range(data.num_nodes):
        ent_type_map[gid] = "other"

# produce counts
from collections import Counter
print("Inferred entity type counts:", Counter(ent_type_map.values()))



Inferred entity type counts: Counter({'other': 94046})


In [13]:
# -------------------------
# Build lists of positive CD pairs and mechanistic edges (comp->gene, gene->disease) FROM TRAIN TRIPLES
# -------------------------
def extract_mechanistic_pairs(gid_triples, ent_type_map, rel_sets=None):
    rel_sets = rel_sets or {}
    comp_d = []
    comp_gene = []
    gene_disease = []
    for h, r, t in gid_triples:
        th = ent_type_map.get(h, "other")
        tt = ent_type_map.get(t, "other")
        rl = r.lower() if isinstance(r, str) else str(r)
        if th=="compound" and tt=="disease":
            comp_d.append((h,t))
        if th=="compound" and tt=="gene":
            comp_gene.append((h,t))
        if th=="gene" and tt=="disease":
            gene_disease.append((h,t))
        # also use relation-name heuristics
        if isinstance(rl, str):
            if any(k in rl for k in ["target","bind","interact"]) and th=="compound" and tt=="gene":
                comp_gene.append((h,t))
            if any(k in rl for k in ["associate","assoc","gene_assoc"]) and th=="gene" and tt=="disease":
                gene_disease.append((h,t))
            if any(k in rl for k in ["treat","indicat"]) and th=="compound" and tt=="disease":
                comp_d.append((h,t))
    # deduplicate
    comp_d = list(set(comp_d)); comp_gene = list(set(comp_gene)); gene_disease=list(set(gene_disease))
    return comp_d, comp_gene, gene_disease

train_pos_cd, train_comp_gene, train_gene_dis = extract_mechanistic_pairs(train_gid_triples, ent_type_map)
val_pos_cd, val_comp_gene, val_gene_dis = extract_mechanistic_pairs(val_gid_triples, ent_type_map)
test_pos_cd, test_comp_gene, test_gene_dis = extract_mechanistic_pairs(test_gid_triples, ent_type_map)

print("Train pos compound->disease:", len(train_pos_cd))
print("Train comp->gene:", len(train_comp_gene), "Train gene->disease:", len(train_gene_dis))



Train pos compound->disease: 0
Train comp->gene: 0 Train gene->disease: 0


In [9]:
# -------------------------
# Negative sampling helpers
# -------------------------
all_diseases = [gid for gid,t in ent_type_map.items() if t=="disease"]
if len(all_diseases)==0:
    print("Warning: no diseases inferred. Check entities.txt format or heuristics.")

def negative_sample_for_compounds(comp_list, num_samples=1):
    negs = []
    for c in comp_list:
        for _ in range(num_samples):
            if len(all_diseases)>0:
                d = random.choice(all_diseases)
                negs.append((c,d))
    return negs

# build train pairs (global gids)
train_comp_list = [c for c,_ in train_pos_cd]
train_negs = negative_sample_for_compounds(train_comp_list, NEG_RATIO)
train_pairs = [(c,d,1) for c,d in train_pos_cd] + [(c,d,0) for c,d in train_negs]
random.shuffle(train_pairs)

val_pairs = [(c,d,1) for c,d in val_pos_cd] + [(c,d,0) for c,d in negative_sample_for_compounds([c for c,_ in val_pos_cd], 1)]
test_pairs = [(c,d,1) for c,d in test_pos_cd] + [(c,d,0) for c,d in negative_sample_for_compounds([c for c,_ in test_pos_cd], 1)]



In [ ]:
# -------------------------
# Models (RGCN, CompGCN-like, GAT, DistMult baseline)
# -------------------------
class RGCNModel(nn.Module):
    def __init__(self, num_nodes, num_rels, hdim=HDIM, outdim=OUTDIM, num_layers=2, dropout=DROPOUT):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, hdim)
        self.convs = nn.ModuleList([RGCNConv(hdim, hdim, num_rels) for _ in range(num_layers)])
        self.dropout = dropout
        self.out = nn.Linear(hdim, outdim)
    def forward(self, x, edge_index, edge_type):
        h = self.emb.weight
        for conv in self.convs:
            h = conv(h, edge_index, edge_type)
            h = F.relu(h); h = F.dropout(h, p=self.dropout, training=self.training)
        return self.out(h)

class CompGCNLike(nn.Module):
    def __init__(self, num_nodes, num_rels, hdim=HDIM, outdim=OUTDIM, num_layers=2, dropout=DROPOUT):
        super().__init__()
        self.ent = nn.Embedding(num_nodes, hdim)
        self.rel = nn.Embedding(num_rels, hdim)
        self.layers = nn.ModuleList([nn.Linear(hdim, hdim) for _ in range(num_layers)])
        self.lin = nn.Linear(hdim, outdim)
        self.dropout = dropout
    def forward(self, x, edge_index, edge_type):
        src = edge_index[0]; dst = edge_index[1]
        E = self.ent.weight; R = self.rel.weight
        msgs = E[src] + R[edge_type]
        h = torch.zeros_like(E).index_add(0, dst, msgs)
        for l in self.layers:
            h = l(h); h = F.relu(h); h = F.dropout(h, p=self.dropout, training=self.training)
        return self.lin(h)

class GATModel(nn.Module):
    def __init__(self, num_nodes, in_channels=64, hidden=128, outdim=OUTDIM, num_heads=2, dropout=DROPOUT):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, in_channels)
        self.gat1 = GATConv(in_channels, hidden//num_heads, heads=num_heads)
        self.gat2 = GATConv(hidden, outdim, heads=1)
        self.dropout = dropout
    def forward(self, x, edge_index, edge_type=None):
        h = self.node_emb.weight
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.gat1(h, edge_index)
        h = F.elu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.gat2(h, edge_index)
        return h

class DistMultBaseline(nn.Module):
    def __init__(self, num_nodes, dim=OUTDIM):
        super().__init__()
        self.E = nn.Embedding(num_nodes, dim)
    def forward(self, *args, **kwargs):
        return self.E.weight



In [ ]:
# -------------------------
# NeighborLoader dataset for mini-batching (full-graph inference when needed)
# -------------------------
# Create a PyG Data object already above. We'll construct a Node-level NeighborLoader for training pairs.
# We train by sampling subgraphs containing both compound and disease nodes from each batch.

# helper to create a list of nodes to sample (compounds + diseases in the batch)
def build_seed_nodes_for_batch(batch_pairs):
    # batch_pairs: list of (comp_gid, dis_gid, label)
    seeds = list(set([p[0] for p in batch_pairs] + [p[1] for p in batch_pairs]))
    return seeds

# -------------------------
# Training utilities (one epoch, per-batch)
# -------------------------
def compute_logits_from_embs(embs, comps, dis):
    # embs: [N, D] tensor
    c = embs[comps]; d = embs[dis]
    logits = torch.sum(F.normalize(c,dim=1) * F.normalize(d,dim=1), dim=1)
    return logits

def path_regularization_loss_from_embeddings(embs, comp_gene_pairs, gene_dis_pairs, device, max_samples=2048):
    # same logic as earlier
    if len(comp_gene_pairs)==0 or len(gene_dis_pairs)==0:
        return torch.tensor(0.0, device=device, requires_grad=True)
    cg = random.sample(comp_gene_pairs, min(len(comp_gene_pairs), max_samples))
    gd = random.sample(gene_dis_pairs, min(len(gene_dis_pairs), max_samples))
    comp_to_genes = defaultdict(list)
    gene_to_dis = defaultdict(list)
    for c,g in cg: comp_to_genes[c].append(g)
    for g,d in gd: gene_to_dis[g].append(d)
    triplets=[]
    for c, genes in comp_to_genes.items():
        for g in genes:
            for d in gene_to_dis.get(g, []):
                triplets.append((c,g,d))
                if len(triplets)>=max_samples: break
            if len(triplets)>=max_samples: break
        if len(triplets)>=max_samples: break
    if len(triplets)==0:
        return torch.tensor(0.0, device=device, requires_grad=True)
    c_idx = torch.tensor([t[0] for t in triplets], dtype=torch.long, device=device)
    g_idx = torch.tensor([t[1] for t in triplets], dtype=torch.long, device=device)
    d_idx = torch.tensor([t[2] for t in triplets], dtype=torch.long, device=device)
    c_v = embs[c_idx]; g_v = embs[g_idx]; d_v = embs[d_idx]
    pred_cd = torch.sum(F.normalize(c_v,dim=1)*F.normalize(d_v,dim=1), dim=1)
    cg_sim = torch.sum(F.normalize(c_v,dim=1)*F.normalize(g_v,dim=1), dim=1)
    gd_sim = torch.sum(F.normalize(g_v,dim=1)*F.normalize(d_v,dim=1), dim=1)
    mech = torch.min(cg_sim, gd_sim)
    margin = 0.1
    loss = F.relu(margin + mech - pred_cd).mean()
    return loss



In [ ]:
# -------------------------
# Full training loop (with NeighborLoader sampling)
# -------------------------
def train_one_model(model, model_name, train_pairs, data, ent_type_map, comp_gene_pairs, gene_dis_pairs, device,
                    epochs=EPOCHS, batch_size=BATCH_SIZE, neighbor_samples=NEIGHBOR_SAMPLES,
                    lr=LR, wd=WEIGHT_DECAY, path_reg_weight=PATH_REG_WEIGHT):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    n = len(train_pairs)
    print(f"[{model_name}] training {n} pairs, epochs={epochs}, batch_size={batch_size}")
    for epoch in range(epochs):
        random.shuffle(train_pairs)
        total_loss = 0.0
        model.train()
        for i in range(0, n, batch_size):
            batch = train_pairs[i:i+batch_size]
            if len(batch)==0: break
            seeds = build_seed_nodes_for_batch(batch)
            # NeighborLoader requires a 'Data' object and node indices; we can sample from entire graph
            loader = NeighborLoader(data, num_neighbors=neighbor_samples, input_nodes=seeds, batch_size=len(seeds))
            # loader yields subgraphs for these seed nodes; but for simplicity we take first subgraph (small)
            sub = next(iter(loader))
            # get mapping from global node idx to local index in sub
            n_id = sub.n_id  # global ids present in sub
            # build index mapping dict
            gid2local = {int(g.item()): idx for idx, g in enumerate(n_id)}
            # prepare batch comp/dis local indices (if present)
            comps_local = []
            dis_local = []
            labels = []
            for c,d,lbl in batch:
                if c in gid2local and d in gid2local:
                    comps_local.append(gid2local[c]); dis_local.append(gid2local[d]); labels.append(lbl)
            if len(comps_local)==0:
                # skip batch if seeds not found due to neighbor truncation
                continue
            comps_t = torch.tensor(comps_local, dtype=torch.long, device=device)
            dis_t = torch.tensor(dis_local, dtype=torch.long, device=device)
            lab_t = torch.tensor(labels, dtype=torch.float32, device=device)
            # run model on subgraph: we need edge_index_sub and edge_type_sub from 'sub'
            edge_index_sub = sub.edge_index.to(device)
            edge_type_sub = getattr(sub, 'edge_type', None)
            if edge_type_sub is not None: edge_type_sub = edge_type_sub.to(device)
            embeddings = model(None, edge_index_sub, edge_type_sub)  # embeddings have rows for sub.n_id order
            # convert comps_local/dis_local -> use embeddings directly because embeddings index align with sub nodes
            logits = compute_logits_from_embs(embeddings, comps_t, dis_t)
            loss = F.binary_cross_entropy_with_logits(logits, lab_t)
            # mechanistic path reg: note embeddings correspond to subgraph indices; but we used global ids in comp_gene_pairs; simplest approach: compute PR using full model embeddings by forwarding on the full graph occasionally (or compute pr using global embeddings if model exposes them)
            # For efficiency, approximate PR using full-graph forward (small overhead)
            if path_reg_weight > 0:
                model.eval()
                with torch.no_grad():
                    full_edge_index = data.edge_index.to(device)
                    full_edge_type = getattr(data, 'edge_type', None)
                    if full_edge_type is not None: full_edge_type = full_edge_type.to(device)
                    full_embs = model(None, full_edge_index, full_edge_type)
                model.train()
                pr_loss = path_regularization_loss_from_embeddings(full_embs, comp_gene_pairs, gene_dis_pairs, device)
                loss = loss + path_reg_weight * pr_loss
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(comps_local)
        avg = total_loss / max(1, len(train_pairs))
        if (epoch+1) % max(1, epochs//4) == 0 or epoch==epochs-1:
            print(f"[{model_name}] epoch {epoch+1}/{epochs} avg_loss={avg:.6f}")
    return model



In [ ]:
# -------------------------
# Ensembles + MC dropout evaluation
# -------------------------
def mc_dropout_predict(model, data, pairs, device, mc_runs=MC_RUNS):
    model.to(device)
    model.train()  # keep dropout active
    embs_samples = []
    with torch.no_grad():
        for _ in range(mc_runs):
            e = model(None, data.edge_index.to(device), getattr(data, 'edge_type', None).to(device) if hasattr(data, 'edge_type') else None)
            embs_samples.append(e.cpu().numpy())
    embs_samples = np.stack(embs_samples, axis=0)  # [mc, N, D]
    means=[]; vars_=[]; labels=[]
    for c,d,lbl in pairs:
        scores=[]
        for s in range(embs_samples.shape[0]):
            vec_c = embs_samples[s, c]; vec_d = embs_samples[s, d]
            sc = float(np.dot(vec_c / (np.linalg.norm(vec_c)+1e-12), vec_d / (np.linalg.norm(vec_d)+1e-12)))
            scores.append(sc)
        means.append(np.mean(scores)); vars_.append(np.var(scores)); labels.append(lbl)
    return np.array(means), np.array(vars_), np.array(labels)

def ensemble_predict(models_list, data, pairs, device, mc_runs=MC_RUNS):
    # models_list: list of model objects (trained). For each model do mc runs; combine means
    all_means = []
    all_vars = []
    for m in models_list:
        means, vars_, labels = mc_dropout_predict(m, data, pairs, device, mc_runs=mc_runs)
        all_means.append(means); all_vars.append(vars_)
    all_means = np.stack(all_means, axis=0)  # [models, N]
    all_vars = np.stack(all_vars, axis=0)
    ensemble_mean = np.mean(all_means, axis=0)
    epistemic = np.var(all_means, axis=0)
    aleatoric = np.mean(all_vars, axis=0)
    total_uncert = np.sqrt(epistemic + aleatoric)
    return ensemble_mean, total_uncert, labels



In [ ]:
# -------------------------
# Run experiments (train ensembles for each model type)
# -------------------------
model_constructors = {
    "rgcn": lambda: RGCNModel(num_nodes, int(data.edge_type.max().item())+1 if hasattr(data,'edge_type') else 8, hdim=HDIM, outdim=OUTDIM, num_layers=2, dropout=DROPOUT),
    "compgc": lambda: CompGCNLike(num_nodes, int(data.edge_type.max().item())+1 if hasattr(data,'edge_type') else 8, hdim=HDIM, outdim=OUTDIM, num_layers=2, dropout=DROPOUT),
    "gat": lambda: GATModel(num_nodes, in_channels=64, hidden=128, outdim=OUTDIM, num_heads=2, dropout=DROPOUT),
    "distmult": lambda: DistMultBaseline(num_nodes, dim=OUTDIM)
}

trained_ensembles = {}
results = {}

for model_name, ctor in model_constructors.items():
    print("\n=== TRAINING ENSEMBLE for", model_name, "===")
    models_list = []
    for en in range(ENSEMBLE_SIZE):
        seed = SEED + en*13 + (0 if model_name=="rgcn" else en)
        set_seed(seed)
        model = ctor()
        print(f"  Training {model_name} member {en+1}/{ENSEMBLE_SIZE} (seed={seed})")
        # train (skip heavy training for distmult baseline - simply learn embeddings via simple objective)
        if model_name == "distmult":
            # train-like: naive embeddings updated via supervised pairs with BCE
            model.to(DEVICE)
            opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            for epoch in range(max(1, EPOCHS//2)):
                random.shuffle(train_pairs)
                total_loss=0.0
                for i in range(0, len(train_pairs), BATCH_SIZE):
                    batch = train_pairs[i:i+BATCH_SIZE]
                    comps = torch.tensor([b[0] for b in batch], dtype=torch.long, device=DEVICE)
                    dis = torch.tensor([b[1] for b in batch], dtype=torch.long, device=DEVICE)
                    labels = torch.tensor([b[2] for b in batch], dtype=torch.float32, device=DEVICE)
                    opt.zero_grad()
                    embs = model()
                    logits = compute_logits_from_embs(embs, comps, dis)
                    loss = F.binary_cross_entropy_with_logits(logits, labels)
                    loss.backward(); opt.step()
                    total_loss += loss.item()*len(batch)
            models_list.append(model)
        else:
            # full GNN model training (neighbor sampling)
            model = train_one_model(model, model_name, train_pairs, data, ent_type_map, train_comp_gene, train_gene_dis, DEVICE,
                                    epochs=EPOCHS, batch_size=BATCH_SIZE, neighbor_samples=NEIGHBOR_SAMPLES, lr=LR, wd=WEIGHT_DECAY,
                                    path_reg_weight=PATH_REG_WEIGHT)
            models_list.append(model)
        # save each member
        torch.save(model.state_dict(), os.path.join(OUT_DIR, f"{model_name}_member{en}.pt"))
    trained_ensembles[model_name] = models_list

    # evaluate ensemble on test_pairs
    print("Evaluating ensemble for", model_name)
    ensemble_mean, ensemble_uncert, labels = ensemble_predict(models_list, data, test_pairs, DEVICE, mc_runs=MC_RUNS)
    try:
        auroc = roc_auc_score(labels, ensemble_mean)
        auprc = average_precision_score(labels, ensemble_mean)
    except:
        auroc = float("nan"); auprc = float("nan")
    results[model_name] = {"auroc": float(auroc), "auprc": float(auprc)}
    print(f"Result {model_name}: AUROC={auroc:.4f}, AUPRC={auprc:.4f}")

# save results
with open(os.path.join(OUT_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
print("Saved results to", os.path.join(OUT_DIR, "results.json"))
print(results)



In [ ]:
# -------------------------
# Produce ranked CSV for held-out diseases: top-k candidate compounds with supporting genes
# -------------------------
TOP_K = 50
# Build a mapping from gene->disease and compound->gene from training mechanistic pairs
gene_to_diseases_map = defaultdict(set)
for g,d in train_gene_dis:
    gene_to_diseases_map[g].add(d)
compound_to_genes_map = defaultdict(set)
for c,g in train_comp_gene:
    compound_to_genes_map[c].add(g)

# For each held-out disease in test set, produce top-K ranked compounds by ensemble_mean, plus supporting genes that chain c->g->d
# First build a list of test diseases that were held out (get distinct diseases from test_pos_cd)
held_out_diseases = sorted(list(set([d for _,d in test_pos_cd])))
if len(held_out_diseases)==0:
    # fall back to diseases appearing in test_pairs positives
    held_out_diseases = sorted(list(set([d for c,d,l in test_pairs if l==1])))
print("Held-out diseases to rank:", len(held_out_diseases))

rank_rows = []
for d in tqdm(held_out_diseases):
    # candidate compounds: all compounds seen in entity types or in training compound list
    candidate_comps = [gid for gid, t in ent_type_map.items() if t=="compound"]
    # compute ensemble predictions for (comp,d) pairs by averaging member predictions (do efficient batched embedding inference)
    # we'll get embeddings by averaging members' full-graph embeddings
    all_embs = []
    for model_name, models_list in trained_ensembles.items():
        # prefer GNN models for ranking (skip distmult baseline)
        for m in models_list:
            emb = m(None, data.edge_index.to(DEVICE), getattr(data,'edge_type',None).to(DEVICE) if hasattr(data,'edge_type') else None)
            emb = emb.cpu().numpy()
            all_embs.append(emb)
    if len(all_embs)==0:
        print("No embeddings available for ranking")
        break
    # average embeddings across all ensemble members
    avg_emb = np.mean(np.stack(all_embs, axis=0), axis=0)  # [members, N, D] -> [N, D]
    # compute similarity scores for all candidate comps to disease d
    scores = []
    vec_d = avg_emb[d]
    for c in candidate_comps:
        vec_c = avg_emb[c]
        sc = float(np.dot(vec_c / (np.linalg.norm(vec_c)+1e-12), vec_d / (np.linalg.norm(vec_d)+1e-12)))
        scores.append((c, sc))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[:TOP_K]
    for rank, (c, sc) in enumerate(scores, start=1):
        # find supporting genes where c->g in compound_to_genes_map and g->d in gene_to_diseases_map
        supporting_genes = []
        for g in compound_to_genes_map.get(c, []):
            if d in gene_to_diseases_map.get(g, set()):
                supporting_genes.append(g)
        rank_rows.append({
            "disease_gid": d,
            "compound_gid": c,
            "score": sc,
            "rank": rank,
            "supporting_genes": ";".join(map(str, supporting_genes)) if supporting_genes else ""
        })

rank_df = pd.DataFrame(rank_rows)
rank_csv_path = os.path.join(OUT_DIR, "ranked_candidates.csv")
rank_df.to_csv(rank_csv_path, index=False)
print("Saved ranked candidates to", rank_csv_path)
print(rank_df.head())

# Done
print("Pipeline finished. Check", OUT_DIR, "for outputs (models, results.json, ranked_candidates.csv).")
